In [1]:
# Imports
from pathlib import Path
import json
import cv2
import numpy as np

In [3]:
# Directory variables
project_root = Path.cwd().parent
raw_dir = project_root / 'data' / 'raw'
video_dir = raw_dir / 'videos'

In [4]:
# Getting the metadata
with (raw_dir / "WLASL_v0.3.json").open("r", encoding="utf-8") as f:
    metadata = json.load(f)

In [5]:
# 26 selected classes
SELECTED_GLOSSES = {
    "thin", "before", "cool", "drink", "go", "who", "help",
    "computer", "cousin", "tall", "short", "candy", "thanksgiving",
    "accident", "bowling", "trade", "what", "basketball", "cold",
    "yes", "change", "pizza", "shirt", "dark", "last", "call"
}

In [6]:
# Get local video ids
local_video_ids = {
    path.stem for path in video_dir.glob("*.mp4")
}
print(f"Local video IDs: {len(local_video_ids)}")

Local video IDs: 11980


In [7]:
# Filter metadata for selected classes
selected_records = [
    record for record in metadata if record["gloss"] in SELECTED_GLOSSES
]
print(f"Selected classes: {len(selected_records)}")

Selected classes: 26


In [8]:
# Collect videos belonging to the selected classes
selected_videos = []

for record in selected_records:
    gloss = record["gloss"]
    for instance in record["instances"]:
        video_id = instance["video_id"]
        if video_id in local_video_ids:
            selected_videos.append({
                "video_id": video_id,
                "gloss": gloss,
                "signer_id": instance.get("signer_id"),
                "split": instance.get("split"),
            })
print(f"Selected available videos: {len(selected_videos)}")

Selected available videos: 345


In [9]:
# Inspect video properties

video_info = selected_videos[0]
video_id = video_info["video_id"]
video_path = video_dir / f"{video_id}.mp4"

print("Video ID:", video_id)
print("ASL word:", video_info["gloss"])
print("Signer:", video_info["signer_id"])
print("Split:", video_info["split"])
print("File exists:", video_path.exists())

Video ID: 69302
ASL word: drink
Signer: 115
Split: val
File exists: True


In [10]:
# Open the video with OpenCV and inspecting properties

# Capture object to read from
capture = cv2.VideoCapture(str(video_path))

# Frames per second
fps = capture.get(cv2.CAP_PROP_FPS)

# Total frame count
frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)

# Total duration of the video
duration = frame_count / fps if fps > 0 else 0

print("Opened:", capture.isOpened())
print("FPS:", fps)
print("Total frames:", int(frame_count))
print("Duration:", round(duration, 2))

capture.release()

Opened: True
FPS: 29.97002997002997
Total frames: 77
Duration: 2.57


In [12]:
# Inspecting all 345 videos

video_stats = []

for video in selected_videos:
    video_id = video["video_id"]
    video_path = video_dir / f"{video_id}.mp4"

    # Capture object to read from
    capture = cv2.VideoCapture(str(video_path))

    # Frames per second
    fps = capture.get(cv2.CAP_PROP_FPS)

    # Total frame count
    frame_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)

    # Total duration of the video
    duration = frame_count / fps if fps > 0 else 0

    video_stats.append({
        "video_id": video_id,
        "fps": fps,
        "frame_count": int(frame_count),
        "duration": round(duration, 2)
    })

    capture.release()

print(f"Videos inspected: {len(video_stats)}")

Videos inspected: 345


In [13]:
# Summarizing the video results

durations = [video["duration"] for video in video_stats]
frame_counts = [video["frame_count"] for video in video_stats]

print("Durations (seconds):")
print(f"Minimum: {round(np.min(durations), 2)}")
print(f"Median: {round(np.median(durations), 2)}")
print(f"Maximum: {round(np.max(durations), 2)}")

print("\nFrame counts:")
print(f"Minimum: {min(frame_counts)}")
print(f"Median: {int(np.median(frame_counts))}")
print(f"Maximum: {max(frame_counts)}")

Durations (seconds):
Minimum: 0.83
Median: 2.2
Maximum: 8.12

Frame counts:
Minimum: 20
Median: 61
Maximum: 195


In [14]:
# Compare sampling rates
total_frames = 77
fps = 29.97

for sampling_fps in [5, 10, 15, 30]:
    sampled_frames = round(total_frames * sampling_fps / fps)
    print(
        f"{sampling_fps} FPS -> "
        f"{sampled_frames} Frames retained"
    )

5 FPS -> 13 Frames retained
10 FPS -> 26 Frames retained
15 FPS -> 39 Frames retained
30 FPS -> 77 Frames retained
